# Laborator 9 – Retele Neuronale Artificiale (ANN) pe MNIST

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
os_env = __import__('os')
os_env.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f"TensorFlow version: {tf.__version__}")

In [ ]:
# Ex 1 – Incarcam si normalizam MNIST
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
print(f"Train: {x_train.shape}, Test: {x_test.shape}")
print(f"Valori pixel inainte: min={x_train.min()}, max={x_train.max()}")

x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32')  / 255.0
print(f"Valori pixel dupa: min={x_train.min():.2f}, max={x_train.max():.2f}")

In [ ]:
# Vizualizare 9 imagini din setul de antrenament
fig, axes = plt.subplots(3, 3, figsize=(6, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i], cmap='gray')
    ax.set_title(f"Cifra: {y_train[i]}")
    ax.axis('off')
plt.suptitle('Exemple MNIST')
plt.tight_layout()
plt.savefig('lab9_mnist_samples.png', dpi=80)
plt.show()

In [ ]:
# Ex 2 – Model Sequential: Flatten + 2 straturi Dense
model = keras.Sequential([
    layers.Flatten(input_shape=(28, 28)),
    layers.Dense(128, activation='relu'),
    layers.Dense(64,  activation='relu'),
    layers.Dense(10,  activation='softmax'),
], name='mnist_ann')

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

In [ ]:
# Ex 3 – Antrenare (5 epoci) si evaluare
history = model.fit(x_train, y_train,
                    epochs=5,
                    batch_size=64,
                    validation_split=0.1,
                    verbose=1)

test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"\nAcuratete pe setul de test: {test_acc:.4f} ({test_acc*100:.2f}%)")

In [ ]:
# Grafic acuratete / pierdere
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(history.history['accuracy'],     label='train')
axes[0].plot(history.history['val_accuracy'], label='val')
axes[0].set_title('Acuratete')
axes[0].set_xlabel('Epoca')
axes[0].legend()

axes[1].plot(history.history['loss'],     label='train')
axes[1].plot(history.history['val_loss'], label='val')
axes[1].set_title('Pierdere (Loss)')
axes[1].set_xlabel('Epoca')
axes[1].legend()
plt.tight_layout()
plt.savefig('lab9_history.png', dpi=80)
plt.show()

In [ ]:
# Ex 4 – Afisam o imagine din setul de test si predictia
idx = 42
img = x_test[idx]
pred = np.argmax(model.predict(img.reshape(1,28,28), verbose=0))
fig, ax = plt.subplots(figsize=(3, 3))
ax.imshow(img, cmap='gray')
ax.set_title(f"Real: {y_test[idx]}, Prezis: {pred}")
ax.axis('off')
plt.savefig('lab9_pred_sample.png', dpi=80)
plt.show()
print(f"Imaginea {idx}: real={y_test[idx]}, prezis={pred}")

In [ ]:
# Ex 5 – Modificare nr neuroni (256 in loc de 128)
model_256 = keras.Sequential([
    layers.Flatten(input_shape=(28, 28)),
    layers.Dense(256, activation='relu'),
    layers.Dense(64,  activation='relu'),
    layers.Dense(10,  activation='softmax'),
])
model_256.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
h256 = model_256.fit(x_train, y_train, epochs=3, batch_size=64,
                     validation_split=0.1, verbose=0)
_, acc256 = model_256.evaluate(x_test, y_test, verbose=0)
print(f"Acuratete model 256 neuroni: {acc256:.4f}")
print("Cu mai multi neuroni modelul poate invata reprezentari mai complexe,")
print("dar riscul de overfitting si timpul de antrenare cresc.")

In [ ]:
# Ex 6 – Numar diferit de epoci (10 epoci)
model_e10 = keras.Sequential([
    layers.Flatten(input_shape=(28, 28)),
    layers.Dense(128, activation='relu'),
    layers.Dense(64,  activation='relu'),
    layers.Dense(10,  activation='softmax'),
])
model_e10.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
h10 = model_e10.fit(x_train, y_train, epochs=10, batch_size=64,
                    validation_split=0.1, verbose=0)
_, acc10 = model_e10.evaluate(x_test, y_test, verbose=0)
print(f"Acuratete dupa 10 epoci: {acc10:.4f}")
print("Mai multe epoci duc in general la acuratete mai buna pana la un punct;")
print("dupa aceea modelul poate incepe sa supraantreneze (overfitting).")

In [ ]:
# Ex 7 – Functia de activare tanh
model_tanh = keras.Sequential([
    layers.Flatten(input_shape=(28, 28)),
    layers.Dense(128, activation='tanh'),
    layers.Dense(64,  activation='tanh'),
    layers.Dense(10,  activation='softmax'),
])
model_tanh.compile(optimizer='adam',
                   loss='sparse_categorical_crossentropy',
                   metrics=['accuracy'])
model_tanh.fit(x_train, y_train, epochs=5, batch_size=64,
               validation_split=0.1, verbose=0)
_, acc_tanh = model_tanh.evaluate(x_test, y_test, verbose=0)
print(f"Acuratete model tanh: {acc_tanh:.4f}")
print("tanh centreaza iesirile in [-1,1], utila in unele contexte,")
print("dar ReLU tinde sa fie mai eficienta in straturile ascunse.")

In [ ]:
# Ex 8 – Fashion MNIST (in loc de MNIST)
(fx_train, fy_train), (fx_test, fy_test) = keras.datasets.fashion_mnist.load_data()
fx_train = fx_train.astype('float32') / 255.0
fx_test  = fx_test.astype('float32')  / 255.0
class_names = ['T-shirt','Trouser','Pullover','Dress','Coat',
               'Sandal','Shirt','Sneaker','Bag','Ankle boot']

model_fashion = keras.Sequential([
    layers.Flatten(input_shape=(28, 28)),
    layers.Dense(128, activation='relu'),
    layers.Dense(64,  activation='relu'),
    layers.Dense(10,  activation='softmax'),
])
model_fashion.compile(optimizer='adam',
                      loss='sparse_categorical_crossentropy',
                      metrics=['accuracy'])
model_fashion.fit(fx_train, fy_train, epochs=5, batch_size=64,
                  validation_split=0.1, verbose=1)
_, facc = model_fashion.evaluate(fx_test, fy_test, verbose=0)
print(f"\nAcuratete Fashion-MNIST: {facc:.4f}")

# Vizualizare sample
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(fx_test[i], cmap='gray')
    pred_f = np.argmax(model_fashion.predict(fx_test[i:i+1], verbose=0))
    ax.set_title(f"R:{class_names[fy_test[i]]}\nP:{class_names[pred_f]}", fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.savefig('lab9_fashion.png', dpi=80)
plt.show()